In [0]:
%run ../Includes/Copy-Datasets

In [0]:
(spark.readStream
	.table("books")
	.createOrReplaceTempView("books_streaming_tmp_vw")
)

In [0]:
%sql
Select * From books_streaming_tmp_vw	

In [0]:
%sql
Select author, count(book_id) as total_books
 From books_streaming_tmp_vw	
 Group By author

In [0]:
%sql
Select *
 From books_streaming_tmp_vw	
 Order By author 

In [0]:
%sql
Create or replace temp view author_counts_tmp_vw As (
	Select author, count(book_id) AS total_books
	From books_streaming_tmp_vw
	Group By author
)


In [0]:
(spark.table("author_counts_tmp_vw")
	.writeStream
	.trigger(processingTime='4 seconds')
	.outputMode("complete")
	.option("checkpointLocation", "dbfs:/mnt/demo/author_counts_checkpoint")
	.table("author_counts")
)

In [0]:
%sql
Select *
From author_counts


In [0]:
%sql
INSERT INTO books
values ("B16", "Hands-On Deep Learning Algorithms with Python", "Sudharsan Ravichandiran", "Computer Science", 25),
        ("B17", "Neural Network Methods in Natural Language Processing", "Yoav Goldberg", "Computer Science", 30),
        ("B18", "Understanding digital signal processing", "Richard Lyons", "Computer Science", 35)

In [0]:
%sql
Select *
From author_counts


In [0]:
%sql
INSERT INTO books
values ("B19", "Introduction to Modeling and Simulation", "Mark W. Spong", "Computer Science", 25),
        ("B20", "Robot Modeling and Control", "Mark W. Spong", "Computer Science", 30),
        ("B21", "Turing's Vision: The Birth of Computer Science", "Chris Bernhardt", "Computer Science", 35)

In [0]:
%sql
Select *
From author_counts

In [0]:
%sql
INSERT INTO books
values ("B16", "Hands-On Deep Learning Algorithms with Python - v1", "Sudharsan Ravichandiran", "Computer Science", 35),
        ("B17", "Neural Network Methods in Natural Language Processing - v1", "Yoav Goldberg", "Computer Science", 40),
        ("B18", "Understanding digital signal processing - v1", "Richard Lyons", "Computer Science", 45)

In [0]:
(spark.table("author_counts_tmp_vw")
	.writeStream
	.trigger(availableNow=True)
	.outputMode("complete")
	.option("checkpointLocation", "dbfs:/mnt/demo/author_counts_checkpoint")
	.table("author_counts")
	.awaitTermination()
)

In [0]:
%sql
Select *
From author_counts